# Week 5 Assignment: A Simplified Transformer Encoder

This notebook extends the attention model from the course material with the components required for a simplified Transformer encoder block.

## Load the IMDB data

The IMDB reviews are already represented as integer word indices. All reviews are padded to the same length so they can be processed in batches.

In [1]:
import keras
from keras.datasets import imdb

max_features = 10000  # vocabulary size
max_len = 250         # words per review

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

x_train = keras.utils.pad_sequences(x_train, maxlen=max_len, padding='post')
x_test = keras.utils.pad_sequences(x_test, maxlen=max_len, padding='post')

print('Training data:', x_train.shape)
print('Test data:', x_test.shape)


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
Training data: (25000, 250)
Test data: (25000, 250)


## Token and positional embedding

The token embedding represents each word as a vector. The positional embedding adds information about the position of each word in the review.

In [2]:
from keras.layers import Layer, Embedding


class TokenAndPositionEmbedding(Layer):
    def __init__(self, seq_len, vocab_size, emb_dim):
        super().__init__()
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=emb_dim)
        self.pos_emb = Embedding(input_dim=seq_len, output_dim=emb_dim)

    def call(self, x_input):
        seq_len = keras.ops.shape(x_input)[-1]
        positions = keras.ops.arange(start=0, stop=seq_len, step=1)
        positions = self.pos_emb(positions)
        x_input = self.token_emb(x_input)
        return x_input + positions


## Simplified Transformer encoder block

The attention output is added to the embedding through a residual connection and normalized. A two-layer feedforward network then processes each position. Its output is added through a second residual connection and normalized again.

The feedforward network returns `emb_dim` units so both residual additions have compatible tensor shapes.

In [3]:
from keras.layers import (
    Input, Dense, GlobalAveragePooling1D, Dropout,
    MultiHeadAttention, Add, LayerNormalization
)
from keras.models import Model

emb_dim = 32
num_heads = 2
feedforward_dim = 64

inputs = Input(shape=(max_len,))
x = TokenAndPositionEmbedding(max_len, max_features, emb_dim)(inputs)

attention_output = MultiHeadAttention(
    num_heads=num_heads,
    key_dim=emb_dim // num_heads
)(x, x)

x = Add()([x, attention_output])
x = LayerNormalization()(x)

feedforward_output = Dense(feedforward_dim, activation='relu')(x)
feedforward_output = Dense(emb_dim)(feedforward_output)

x = Add()([x, feedforward_output])
x = LayerNormalization()(x)

x = GlobalAveragePooling1D()(x)
x = Dropout(0.5)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 250)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_and_position_embedding  │ (None, 250, 32)           │         328,000 │ input_layer[0][0]          │
│ (TokenAndPositionEmbedding)   │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ multi_head_attention          │ (None, 250, 32)           │           4,224 │ token_and_position_embedd… │
│ (MultiHeadAttention)          │                           │                 │ token_and_position_embedd… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add (Add)                     │ (None, 250, 32)           │               0 │ token_and_position_embedd… │
│                               │                           │                 │ multi_head_attention[0][0] │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ layer_normalization           │ (None, 250, 32)           │              64 │ add[0][0]                  │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 250, 64)           │           2,112 │ layer_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 250, 32)           │           2,080 │ dense[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_1 (Add)                   │ (None, 250, 32)           │               0 │ layer_normalization[0][0], │
│                               │                           │                 │ dense_1[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ layer_normalization_1         │ (None, 250, 32)           │              64 │ add_1[0][0]                │
│ (LayerNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling1d      │ (None, 32)                │               0 │ layer_normalization_1[0][… │
│ (GlobalAveragePooling1D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 32)                │               0 │ global_average_pooling1d[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 1)                 │              33 │ dropout_1[0][0]            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 336,577 (1.28 MB)

 Trainable params: 336,577 (1.28 MB)

 Non-trainable params: 0 (0.00 B)

## Train and test the model

Binary cross-entropy and one sigmoid output are suitable for the two IMDB sentiment classes.

In [4]:
model.fit(x_train, y_train, epochs=5, batch_size=32)

print(f'Test accuracy = {model.evaluate(x_test, y_test)[1]:.4f}')


Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 72s 81ms/step - accuracy: 0.8000 - loss: 0.4140
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 70s 89ms/step - accuracy: 0.9154 - loss: 0.2228
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 69s 87ms/step - accuracy: 0.9386 - loss: 0.1665
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 65s 83ms/step - accuracy: 0.9552 - loss: 0.1295
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 62s 80ms/step - accuracy: 0.9655 - loss: 0.1043
782/782 ━━━━━━━━━━━━━━━━━━━━ 21s 25ms/step - accuracy: 0.8507 - loss: 0.4738
Test accuracy = 0.8507


## Conclusion

The model contains token and positional embeddings, multi-head self-attention, a feedforward network, two residual connections, two normalization layers, global average pooling, and a sigmoid output. These are the components requested for the simplified Transformer encoder.